# Retail Decision Lab

Reproduce the dashboard analysis with shared modules. See `docs/DATA.md` for the provisional sales grain and missing data qualifications. This notebook supersedes the archived coursework outputs.

In [ ]:
from pathlib import Path
import sys

# Works from the repository root or the notebooks directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "retail").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook inside the cloned repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from retail.data import load_data, audit_data, daily_sales
from retail.forecast import run_evaluation
from retail.inventory import InventoryAssumptions, compare_inventory

## Audit before aggregation

Distinct C520 records are a provisional interpretation of repeated sales. Conflicting discount flags are audited, not used as forecast features.

In [ ]:
raw = load_data()
audit, product_audit = audit_data(raw)
display(audit)
display(product_audit.sort_values("raw_sales", ascending=False).head(10))

## Chronological evaluation

The assortment is chosen before validation. Three expanding validation windows choose the model and baseline. The final 20% is used for reporting, with missing outcomes and incomplete lag windows excluded.

In [ ]:
daily = daily_sales(raw, mode="distinct")
analysis = run_evaluation(daily)
display(analysis["splits"])
display(analysis["excluded"])
display(analysis["coverage"])
display(analysis["selection"])
display(analysis["test"])

## Inspect a product

MAE alone can hide systematic overprediction. Inspect bias and the actual-versus-predicted path.

In [ ]:
product_id = analysis["selection"].product_id.iloc[0]
selected = analysis["selection"].set_index("product_id").loc[product_id]
predictions = analysis["predictions"].query("product_id == @product_id")
curves = predictions.pivot(index="date", columns="model", values="prediction")
curves["Observed sales"] = predictions.drop_duplicates("date").set_index("date").actual
curves.plot(figsize=(12, 4), title=selected["product"]);

## Simulate inventory with stated assumptions

Each policy starts with fresh stock at its initial target. Orders arrive after lead time, stock expires, and unmet sales are lost. Buffer quantiles are calibrated on validation horizon errors; their labels do not guarantee service.

In [ ]:
assumptions = InventoryAssumptions(lead_days=2, shelf_days=3, holding_cost=0.01, waste_cost=1.0, lost_sale_cost=2.0)
scenarios, traces = compare_inventory(analysis, product_id, assumptions)
display(scenarios)
policy = next(iter(traces))
traces[policy][["demand", "closing_stock", "lost_sales", "waste"]].plot(figsize=(12, 4), title=policy);

## Interpretation and next experiment

Compare the selected model with its validation-selected baseline. Explain whether changing the buffer improves fill rate at an acceptable waste cost under the chosen assumptions. Do not claim real savings or causal price effects. Confirm the sales grain and permissions with the source owner, then evaluate on a fresh store-period.